# CS224N · Lecture 19 — Open Problems and Discussion

**Slides:** [cs224n-2024-lecture19-open-problems.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-2024-lecture19-open-problems.pdf) (Tatsunori Hashimoto)  
**Companion:** the Lecture 18 notebook on *NLP, linguistics, and philosophy* (Manning) covers the course recap, shortcuts in sequence models, symbolic vs. neural systems, meaning, and AI risks. This notebook goes deeper on the **open problems**, each with a small experiment.

| § | Open problem | You will implement / measure |
|---|---|---|
| 1 | Important ideas to remember | — |
| 2 | **Generalization**: shortcuts, ICL with random labels, systematic generalization, robustness | **"Accuracy on the line"**: in- vs. out-of-distribution accuracy across many models |
| 3 | **Analysis and interpretation** | 🔎 **Sparse autoencoders** recover superposed features |
| 4 | **Multilingual** | Tokenization cost of the same sentence across languages |
| 5 | **Evaluation** | — |
| 6 | **Domains and modalities** | — |
| 7 | **Fairness and social impact** | **WEAT**; 🔎 why "hard debiasing" doesn't remove bias |
| 8 | **Sustainability** | 🔎 A training energy/carbon estimate, checked against Llama 2's reported numbers |
| 9 | Wrapping up | — |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, random, collections
from scipy.stats import norm

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))
def train_logreg(X, y, steps=400, lr=0.5, l2=1e-3):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b); g = p - y
        w -= lr * (X.T @ g / len(y) + l2 * w); b -= lr * g.mean()
    return w, b

---
## 1. Important ideas from CS224N to remember

* **Idea 1: dense representations and distributional semantics.** Predict neighboring words using a word-vector representation (L1–2).
* **Idea 2: depth and neural networks.** The challenges of training large, deep nets; many tricks like residual connections (L3, L6, L8).
* **Idea 3: sequence models and computational efficiency.** Vanishing gradients and information propagation; the parallelization and dependency benefits of Transformers (L5–8, L18).
* **Idea 4: language modeling and pretraining.** Language modeling as a "universal" pretraining task; careful and smart scaling to internet-wide data (L9–10, L12, L15).

**Open problems (a quick and incomplete overview):** generalization · analysis and understanding · multilinguality · evaluation · domains and modalities · fairness and social impact. *Your final-project model is a black box summarized by "Accuracy: ___ %". What's inside, and how far does it generalize?*

---
## 2. Generalization

### 2.1 Do models understand our tasks?
There are many instances of models using **shortcuts** over deep understanding: the NLI negation artifact (*"The economy could be still better"* → *"The economy has **never** been better"* is labeled contradiction because of the negation word; L11 §2.3 measured a hypothesis-only classifier exploiting exactly this).

### 2.2 How much do models really generalize?
Even modern LLMs seem to leverage surface cues: **are we just finding better shortcuts?** A striking example: **in-context learning with random labels does almost as well as with real labels** (Min et al. 2022, *Rethinking the Role of Demonstrations*). Replacing the gold labels in few-shot demonstrations with random ones barely hurts on many classification tasks; what matters more is the **label space**, the **distribution of the input text**, and the **format**. The demonstrations seem to tell the model *which task and format*, more than they teach the input–label mapping (consistent with the "task location" view of ICL in L10 §1.2; larger models do use the labels more, Wei et al. 2023).

### 2.3 How do we make models go beyond the training data?
Can we generalize in **humanlike ways, from little data**? **SCAN** (Lake & Baroni 2018) maps commands to action sequences (*"jump twice"* → `JUMP JUMP`, *"walk left after run"* → `RUN LTURN WALK`). If a model sees *"jump"* only in isolation during training, can it understand *"jump twice"*, *"jump around left"*? Humans do this effortlessly (**systematic generalization**, L18 §5.1). Standard seq2seq models trained on SCAN's "add primitive jump" split got almost none of these right, despite near-perfect accuracy on random splits. (Large pretrained LMs do much better, but compositional generalization benchmarks like COGS remain challenging.)

### 2.4 Is there much beyond optimizing IID error?
**Despite our best intuitions, the best models on average are also the most robust.** Across hundreds of models, accuracy under distribution shift is strongly **linearly correlated** with in-distribution accuracy, after a probit transform (Taori et al. 2020; Miller et al. 2021, *Accuracy on the Line*). Interventions designed specifically for robustness rarely beat that line; improving average accuracy (more data, bigger models, better pretraining) moves models along it. Let's see the phenomenon in a simulation: many models of varying quality (different training-set sizes, regularization, and feature sets), evaluated in-distribution and under a covariate shift:

In [ ]:
r = np.random.default_rng(0)
d = 30
w_true = r.normal(size=d)
def sample(n, shift=False):
    X = r.normal(size=(n, d))
    if shift:
        X = 1.6 * X + 0.4                                         # covariate shift: wider spread and a mean offset
        X += 0.8 * r.normal(size=X.shape)                         # plus input noise
    y = (X @ w_true + r.normal(0, 2.0, n) > 0).astype(float)
    return X, y

X_id, y_id = sample(5000); X_ood, y_ood = sample(5000, shift=True)
results = []
for n_train in [15, 30, 60, 120, 250, 500, 2000]:
    for l2 in [1e-3, 1e-1, 1.0]:
        for n_feat in [10, 20, 30]:                                # models that see only some of the features
            feats = r.choice(d, n_feat, replace=False)
            Xt, yt = sample(n_train)
            w_, b_ = train_logreg(Xt[:, feats], yt, l2=l2)
            acc = lambda X, y: np.mean(((X[:, feats] @ w_ + b_) > 0) == y)
            results.append((acc(X_id, y_id), acc(X_ood, y_ood)))
res = np.array(results)
pid, pood = norm.ppf(np.clip(res[:, 0], 0.01, 0.99)), norm.ppf(np.clip(res[:, 1], 0.01, 0.99))
slope, intercept = np.polyfit(pid, pood, 1)
r2 = np.corrcoef(pid, pood)[0, 1] ** 2
print(f"{len(res)} models: probit(OOD accuracy) vs probit(ID accuracy): slope {slope:.2f}, R^2 = {r2:.3f}")
plt.figure(figsize=(5, 4)); plt.scatter(pid, pood, s=12)
xs = np.linspace(pid.min(), pid.max(), 10); plt.plot(xs, slope * xs + intercept, "r--")
plt.xlabel("probit(in-distribution accuracy)"); plt.ylabel("probit(out-of-distribution accuracy)")
plt.title("Accuracy on the line (simulation)"); plt.show()

The models fall close to a line: knowing a model's in-distribution accuracy predicts its accuracy under this shift. That's why "just make the model better on average" has been such an effective robustness strategy, and also why a model **far above the line** is interesting (it has gained *effective robustness*). The line isn't universal: some shifts (e.g., adversarial ones, or shifts that break spurious correlations the model relies on) don't follow it.

---
## 3. Analysis and interpretation: what makes NLP systems work?

*xkcd 1838: "This is your machine learning system? Yup! You pour the data into this big pile of linear algebra, then collect the answers on the other side."* We summarize models with one or a handful of accuracy numbers. **What do they learn? Why do they succeed and fail?**

* **Old results already show interpretable latent units** (Karpathy et al. 2016): individual cells of a character-level LSTM language model track things like **being inside quotes** or the position in a line. (L5 §7.1 found a linear probe that reads the subject's number from an RNN.)
* **Can we build interpretable, but performant models?** **Backpack language models** (Hewitt et al. 2023): each word has several **sense vectors**, and the model's output is a **weighted (log-linear) sum of the sense vectors** of the words in context, with weights computed by a Transformer. Because the output is a linear function of the sense vectors, you can **intervene** predictably, e.g., edit one sense vector to reduce a gender bias, and the change propagates in every context.
* **Can understanding help find the next Transformer?** What can and can't be learned via language-model pretraining? What does deep learning struggle to do? What do neural models tell us about language? How are our models affecting people, and transferring power?

### 3.1 🔎 Sparse autoencoders and superposition
A major thread in recent **mechanistic interpretability**: models seem to represent **more features than they have dimensions**, by storing them as nearly-orthogonal directions that are rarely active at the same time (**superposition**; Elhage et al. 2022). Then individual neurons look "polysemantic", and PCA-style analysis fails (it finds at most $d$ directions). **Sparse autoencoders** (SAEs; Bricken et al. 2023, Cunningham et al. 2023, and since scaled to large production models) learn an **overcomplete dictionary** with a sparsity penalty:
$$f = \text{ReLU}(W_e(a - b_d) + b_e),\qquad \hat a = W_d f + b_d,\qquad \mathcal{L} = \|a - \hat a\|^2 + \lambda\|f\|_1$$
We simulate activations built from **48 sparse ground-truth features squeezed into 16 dimensions**, and check whether an SAE recovers the true feature directions:

In [ ]:
r = np.random.default_rng(0)
d_act, n_feat, n = 16, 48, 20000
D_true = r.normal(size=(d_act, n_feat)); D_true /= np.linalg.norm(D_true, axis=0)      # true feature directions
Z = (r.random((n, n_feat)) < 0.04) * r.uniform(0.5, 1.5, (n, n_feat))                   # sparse feature activations
A = Z @ D_true.T                                                                         # superposed activations (n, 16)

def train_sae(lam, m=64, steps=4000, lr=3e-3, seed=0):
    r_ = np.random.default_rng(seed)
    We = r_.normal(0, 0.3, (m, d_act))
    P = {"We": We, "be": np.zeros(m), "Wd": We.T / np.linalg.norm(We.T, axis=0), "bd": np.zeros(d_act)}
    mom = {k: np.zeros_like(v) for k, v in P.items()}; vel = {k: np.zeros_like(v) for k, v in P.items()}
    for t in range(1, steps + 1):
        a = A[r_.integers(0, n, 512)]
        pre = (a - P["bd"]) @ P["We"].T + P["be"]; f = np.maximum(pre, 0)
        a_hat = f @ P["Wd"].T + P["bd"]
        da = 2 * (a_hat - a) / len(a)                                                        # reconstruction gradient
        g = {"Wd": da.T @ f, "bd": da.sum(0)}
        dpre = (da @ P["Wd"] + lam * (f > 0) / len(a)) * (pre > 0)                            # + L1 penalty gradient
        g["We"] = dpre.T @ (a - P["bd"]); g["be"] = dpre.sum(0); g["bd"] -= (dpre @ P["We"]).sum(0)
        for k in P:
            mom[k] = 0.9 * mom[k] + 0.1 * g[k]; vel[k] = 0.999 * vel[k] + 0.001 * g[k] ** 2
            P[k] -= lr * (mom[k] / (1 - 0.9 ** t)) / (np.sqrt(vel[k] / (1 - 0.999 ** t)) + 1e-8)
        P["Wd"] /= np.linalg.norm(P["Wd"], axis=0, keepdims=True)                             # unit-norm dictionary atoms
    f_all = np.maximum((A - P["bd"]) @ P["We"].T + P["be"], 0)
    rel_err = np.mean(((f_all @ P["Wd"].T + P["bd"] - A) ** 2).sum(1)) / np.mean((A ** 2).sum(1))
    return P["Wd"], np.mean((f_all > 1e-6).sum(1)), rel_err

def recovered(dictionary, thresh=0.9):
    cos = np.abs(D_true.T @ (dictionary / np.linalg.norm(dictionary, axis=0)))           # (true features, atoms)
    return np.mean(cos.max(1) > thresh), np.median(cos.max(1))

_, _, Vt = np.linalg.svd(A - A.mean(0), full_matrices=False)
pca_frac, pca_med = recovered(Vt.T)
print(f"{n_feat} true features in {d_act} dimensions; true active features per input: {np.mean((Z > 0).sum(1)):.1f}")
print(f"PCA ({d_act} components): {pca_frac:.0%} of true features matched (cosine > 0.9), median best cosine {pca_med:.2f}")
print(f"{'SAE, lambda':>12s} {'features matched':>17s} {'active latents':>15s} {'reconstruction error':>21s}")
for lam in [0.05, 0.3, 1.0, 3.0]:
    Wd_, active, rel_err = train_sae(lam)
    frac, med = recovered(Wd_)
    print(f"{lam:12.2f} {frac:17.0%} {active:15.1f} {rel_err:21.1%}")

PCA can't represent 48 directions in 16 dimensions. A sparse autoencoder can recover nearly all of them, **but only with the right sparsity penalty**, and the sweep shows the central tradeoff of real SAE work:
* **λ too small:** latents stay dense (many active per input) and entangled; they don't line up with the true features.
* **λ well chosen:** about as many active latents per input as there are true active features, and the dictionary atoms match the true features. The price is a higher reconstruction error (the L1 penalty shrinks activations; practical SAEs counter this with variants like TopK or gated SAEs).
* **λ too large:** most latents die, and the SAE reconstructs almost nothing.

In real models, there's no ground truth to check against, so practitioners judge SAEs by reconstruction loss, sparsity, the downstream loss when the SAE's reconstruction replaces the activations, and human or automated interpretability of the features. Open problems remain: features that don't neatly decompose, dependence on dictionary size and λ, and whether SAE features are the "right" units of analysis at all.

---
## 4. Multilingual

**Significant gaps between high-resource languages (English) and others (e.g., Telugu), even for the best models.** **Data drives pretraining, so gaps in data availability lead to performance gaps.** How can we close this?
* **Working with extremely low-resource languages:** most languages do not have machine-readable written text; many may become extinct; and there's little for-profit motive to serve them: a **vicious feedback loop** (Adda et al. 2016).
* **Data quality is very variable** in multilingual corpora: in an audit of large web-mined multilingual datasets (Kreutzer et al. 2022), for **20% of languages less than 50% of the sampled data was correct** (wrong language, non-linguistic content, or poor translations). Participatory efforts like Masakhane (Nekoto et al. 2020) build data and models with native-speaker communities.
* **How can we get better multilingual training data?** It's very difficult to scale things like MT or instruction-tuning data: automatic (alignment, mining, translation) approaches vs. crowdsourced approaches.

### 4.1 🔎 Tokenization makes the gap worse
A tokenizer learned mostly from English text (L9 §1) splits other languages into many more tokens, so the **same content costs more** (more compute and money per sentence, and less fits in the context window). Byte-level tokenizers make it worse for non-Latin scripts: UTF-8 uses **3 bytes** per character for Devanagari, Telugu, and Chinese. We train a small byte-level BPE on English text and count tokens for the same short sentences:

In [ ]:
english = ("hello how are you today . thank you very much for your help . the weather is nice and the students are happy . "
           "we are learning about language models and how they work . thank you for coming to the lecture today . "
           "how are you doing with your final project . the model is very good at english text . ") * 30
def byte_bpe(text, n_merges):
    words = collections.Counter(text.split())
    vocab = {tuple(bytes([b_]) for b_ in w.encode("utf-8")): c for w, c in words.items()}
    merges = []
    for _ in range(n_merges):
        pairs = collections.Counter()
        for w, c in vocab.items():
            for a_, b_ in zip(w, w[1:]):
                pairs[(a_, b_)] += c
        if not pairs:
            break
        best = max(pairs, key=pairs.get); merges.append(best)
        new = {}
        for w, c in vocab.items():
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and (w[i], w[i + 1]) == best:
                    out.append(w[i] + w[i + 1]); i += 2
                else:
                    out.append(w[i]); i += 1
            new[tuple(out)] = c
        vocab = new
    return merges

def tokenize(text, merges):
    rank = {m_: i for i, m_ in enumerate(merges)}; n_tok = 0
    for w in text.split():
        sym = [bytes([b_]) for b_ in w.encode("utf-8")]
        while True:
            cand = [(rank.get((a_, b_), 1e9), i) for i, (a_, b_) in enumerate(zip(sym, sym[1:]))]
            if not cand or min(cand)[0] == 1e9:
                break
            _, i = min(cand); sym = sym[:i] + [sym[i] + sym[i + 1]] + sym[i + 2:]
        n_tok += len(sym)
    return n_tok

merges = byte_bpe(english, 200)
sentences = {"English": "hello , how are you ?", "Spanish": "hola , ¿ cómo estás ?", "Hindi": "नमस्ते , आप कैसे हैं ?",
             "Chinese": "你好 ， 你好吗 ？", "Telugu": "హలో , మీరు ఎలా ఉన్నారు ?"}
base = tokenize(sentences["English"], merges)
print(f"{'language':9s} {'characters':>10s} {'UTF-8 bytes':>12s} {'tokens':>7s} {'cost vs English':>16s}")
for lang, s in sentences.items():
    n_t = tokenize(s, merges)
    print(f"{lang:9s} {len(s.replace(' ', '')):10d} {len(s.replace(' ', '').encode('utf-8')):12d} {n_t:7d} {n_t / base:15.1f}x")

The same greeting costs several times more tokens in Hindi, Chinese, or Telugu than in English, under a tokenizer that saw only English. Production tokenizers are trained on multilingual data and do much better, but measured disparities of 2–5× (sometimes far more) for under-represented languages are common (Petrov et al. 2023, *Language Model Tokenizers Introduce Unfairness Between Languages*). Speakers of those languages pay more, wait longer, and get shorter effective contexts.

---
## 5. Evaluation and comparison

**Benchmarks and how we evaluate drive the progress of the field** (L11).
* **Evaluating chatbots and open-domain systems:** we know our evals are broken in some known ways (**length**, L11 §5.3), but we have **no idea what other ways they're broken**.
* **What's the gold standard in evaluation?** Human preference (Chatbot Arena) is expensive, slow, and not representative of every use; LLM judges inherit their own biases.
* **How do we maintain benchmark integrity?** Pretraining on the web makes it increasingly difficult to build **hidden test sets** (contamination, L11 §7.2).
* **How do we evaluate things like interpretability?** Increasingly, many things in research are **qualitative**. How do we evaluate those?

---
## 6. Making NLP work in high-impact domains, and other modalities

NLP systems (and LLMs) are going **from the lab to the real world**:
* **Bio/clinical NLP:** enormous potential (and risks) in medical and basic-science settings: note-taking, QA, "curbside consults".
* **Legal NLP:** systems that understand and assist users with legal questions might address the **"justice gap"** (most low-income people's civil legal problems get no or inadequate legal help), but they must understand complex jargon and be reliable.
* **Scientific NLP:** some emerging successes, e.g., **FunSearch** (Romera-Paredes et al., *Nature* 2023), where an LLM proposes programs and an automated evaluator scores them, finding new constructions in combinatorics (the cap set problem); but also challenges, e.g., **Galactica** (Meta, 2022), a science LM whose public demo was withdrawn within days after it produced authoritative-sounding but fabricated scientific text.
* **Education:** NLP systems have the potential to unlock **Bloom's two-sigma effect** (Bloom 1984: students tutored one-on-one performed about two standard deviations better than students taught conventionally), but they also really shake up education (assessment, homework, academic integrity).

**NLP + other modalities:** excitement about text + X (vision, code, ...). **Image–text models finally work** (Flamingo 2022; LLaVA 2023, L14 §7), **but they're less deep than expected**: they often rely on language priors and shallow visual grounding. Is there a deeper synthesis that will yield major gains?

---
## 7. Fairness and social impact

The deployment of NLP systems makes fairness and external impact concerns even more important: **making models treat people fairly**, and **building less stereotyped representations**.
* **How to quantify biases and stereotypes:** StereoSet, BBQ (L17 §2.1).
* **How to evaluate and mitigate:** biases in **word embeddings** (Caliskan, Bryson & Narayanan, *Science* 2017); disparities for low-resourced languages and dialects (L17 §3).
* **What are the unintended impacts of LLM alignment?** Ryan, Held & Yang (2024) prompted the **Starling-7B reward model** with *"User: Where are you from? Assistant: I am from {country}."* It assigned **higher rewards to English-speaking Western nations** and lower rewards to countries in the Middle East and Africa: biases in preference data become biases in what the policy is optimized toward.

### 7.1 The Word Embedding Association Test (WEAT; Caliskan et al. 2017)
WEAT adapts the psychologists' Implicit Association Test to word vectors. For two sets of **target** words $X, Y$ (e.g., career vs. family terms) and two sets of **attribute** words $A, B$ (e.g., male vs. female names):
$$s(w, A, B) = \text{mean}_{a\in A}\cos(w, a) - \text{mean}_{b\in B}\cos(w, b),\qquad d = \frac{\text{mean}_{x\in X}s(x, A, B) - \text{mean}_{y\in Y}s(y, A, B)}{\text{std}_{w\in X\cup Y}\,s(w, A, B)}$$
with a **permutation test** (re-split $X\cup Y$ at random) for significance. Caliskan et al. found large effect sizes for many human-documented associations in GloVe and word2vec. We plant an association in synthetic embeddings and measure it:

In [ ]:
r = np.random.default_rng(0)
dim = 50
g = r.normal(size=dim); g /= np.linalg.norm(g)                      # a "gender" direction
h = r.normal(size=dim); h -= (h @ g) * g; h /= np.linalg.norm(h)     # a correlated direction (e.g., topic/context)
def word(bias, leak=0.8):
    return 0.6 * bias * g + 0.6 * leak * bias * h + r.normal(0, 0.3, dim)   # bias shows up along g AND leaks along h
X_career = [word(+1) for _ in range(8)]; Y_family = [word(-1) for _ in range(8)]
A_male = [1.2 * g + r.normal(0, 0.3, dim) for _ in range(8)]; B_female = [-1.2 * g + r.normal(0, 0.3, dim) for _ in range(8)]

def cos(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))
def s_assoc(w, A, B):
    return np.mean([cos(w, a) for a in A]) - np.mean([cos(w, b) for b in B])
def weat(X, Y, A, B, n_perm=5000, r_=np.random.default_rng(1)):
    sx, sy = [s_assoc(x, A, B) for x in X], [s_assoc(y, A, B) for y in Y]
    d_eff = (np.mean(sx) - np.mean(sy)) / np.std(sx + sy, ddof=1)
    stat = np.sum(sx) - np.sum(sy); allv = np.array(sx + sy); count = 0
    for _ in range(n_perm):
        p_ = r_.permutation(len(allv)); count += (allv[p_[:len(sx)]].sum() - allv[p_[len(sx):]].sum()) >= stat
    return d_eff, count / n_perm

d_eff, p_val = weat(X_career, Y_family, A_male, B_female)
print(f"WEAT before debiasing: effect size d = {d_eff:.2f}, permutation p = {p_val:.4f}")

### 7.2 🔎 "Lipstick on a pig": why projecting out a bias direction isn't enough
**Hard debiasing** (Bolukbasi et al. 2016) removes the component along the identified bias direction $g$ from every (non-gender-definitional) word: $w \leftarrow w - (w^\top g)\,g$. WEAT along $g$ then drops to near zero. But **Gonen & Goldberg (2019)** showed the bias information is **still there**: words that were originally biased still cluster together, and a classifier can recover each word's original bias from the "debiased" vectors, because bias also lives in correlated directions. In our embeddings the bias leaks along $h$:

In [ ]:
debias = lambda w: w - (w @ g) * g
d_eff2, p_val2 = weat([debias(x) for x in X_career], [debias(y) for y in Y_family], A_male, B_female)
print(f"WEAT after projecting out g: effect size d = {d_eff2:.2f}, p = {p_val2:.3f}   <- the measured bias is gone...")

# ...but is the information gone? 600 occupation-like words with a known original bias score
biases = r.normal(size=600)
words_ = np.array([word(b_) for b_ in biases]); words_db = np.array([debias(w) for w in words_])
labels = (biases > 0).astype(float)
w_c, b_c = train_logreg(words_db[:400], labels[:400], steps=1000)
acc = np.mean(((words_db[400:] @ w_c + b_c) > 0) == labels[400:])
print(f"a linear classifier recovers each word's ORIGINAL bias direction from the debiased vectors: {acc:.1%} accuracy (chance 50%)")

The test we chose to measure (association along $g$) improved, and the underlying information didn't go anywhere. **Mitigation measured only by the metric it optimized can be "lipstick on a pig."** Evaluate bias with multiple methods (WEAT, downstream behavior, probing), and in the LLM era, mostly on **downstream behavior** (BBQ, counterfactual prompts, L17 §2.1).

**Social aspects of NLP** (Hovy & Yang 2021): formulating research questions and techniques around **social awareness**: the **speakers** (low-resourced languages and dialects, vulnerable populations), **culture and ideology** (whose culture and values get represented?), and **norms and context** (when and where behaviors are expected). **Implicit knowledge in NLP models:** temporal, physical, and social **commonsense** (e.g., COMET, Bosselut et al. 2019), and **"theory of mind"** (reasoning about others' beliefs; Sap et al. 2022 found LLMs' social intelligence limited).

---
## 8. Sustainability and energy

**Big NLP systems use a lot of energy. Can we make things more efficient?** (L12 and L18 are the technical answers.) 🔎 A back-of-the-envelope estimate you should be able to do for any training run: compute ≈ $6ND$ FLOPs (L9), GPU throughput = peak × MFU (L12), energy = GPU-hours × power × PUE (data-center overhead), emissions = energy × grid carbon intensity. We check it against the numbers Meta reported for **Llama 2 7B** (2 trillion tokens on A100-80GB GPUs with a 400 W power draw): **184,320 GPU-hours** and **31.22 tCO₂eq**.

In [ ]:
def training_footprint(n_params, n_tokens, peak_flops=312e12, mfu=0.4, gpu_kw=0.4, pue=1.0, kg_co2_per_kwh=0.423):
    flops = 6 * n_params * n_tokens
    gpu_hours = flops / (peak_flops * mfu) / 3600
    energy_mwh = gpu_hours * gpu_kw * pue / 1000
    return gpu_hours, energy_mwh, energy_mwh * kg_co2_per_kwh            # tonnes CO2 (MWh x kg/kWh = t)

gh, mwh, t = training_footprint(7e9, 2e12)
print(f"Llama 2 7B estimate: {gh:,.0f} GPU-hours, {mwh:,.0f} MWh, {t:.1f} tCO2eq   (reported: 184,320 GPU-hours, 31.22 tCO2eq)")
for name, n_p, n_t in [("70B on 2T tokens", 70e9, 2e12), ("7B on 15T tokens", 7e9, 15e12)]:
    gh, mwh, t = training_footprint(n_p, n_t)
    print(f"{name:18s}: {gh:>12,.0f} GPU-hours, {mwh:>8,.0f} MWh, {t:>8,.0f} tCO2eq (same assumptions)")
print("(assumptions: A100 BF16 peak, 40% MFU, 400 W per GPU, PUE folded into the carbon intensity; real runs vary widely by grid)")

The simple formula lands within a few percent of the reported Llama 2 numbers: training cost is predictable from parameters and tokens. Two things the formula hides: **inference** can dominate the lifetime footprint of a widely used model (every query costs ~$2N$ FLOPs per token), and **grid carbon intensity** varies by more than 10× across locations and times, so *where and when* you train matters as much as how efficiently.

---
## 9. Wrapping up

Thanks for sticking with the lectures and the course!
* **Key ideas:** distributed representations, building neural-network-based NLP systems, pretraining, adaptation.
* **Major opportunities:** NLP systems work in ways that enable real-world applications. It's exciting times for the field.
* **Remaining open questions:** how do we solve some of the lingering and complex problems, like **generalization** or **evaluation**?

🔎 **How to keep learning after this series**
* **Reproduce, don't just read:** pick a recent paper with released code, reproduce one table on a small scale, then change one thing (the exercises throughout these notebooks are designed for that).
* **Build the measurement first:** most of the corrections in this series came from checking a claim against an experiment. Make that a habit: baselines, ablations, seeds, confidence intervals (L7, L11).
* **Read across the stack:** a modern ML engineer needs the math (L1–8), the systems (L12, L18), the data (L9, L15, L17), and the evaluation (L11), not just the model.
* **Follow primary sources:** papers (ACL Anthology, arXiv), model and dataset cards, technical reports; treat leaderboards and benchmarks critically (L11, §5 here).

### Summary of this notebook
| Open problem | What we measured |
|---|---|
| Generalization | Shortcuts (NLI), ICL with random labels, SCAN; **accuracy on the line**: OOD accuracy tracks ID accuracy across models |
| Analysis | Interpretable units, Backpacks; 🔎 **SAEs** recover superposed features that PCA can't, if the sparsity penalty is right |
| Multilingual | Data gaps and quality; 🔎 the same sentence costs several times more tokens outside English |
| Evaluation | Known (length) and unknown biases; integrity under web-scale pretraining; qualitative research |
| Domains & modalities | Clinical, legal, science (FunSearch vs. Galactica), education (two-sigma); shallow multimodal grounding |
| Fairness | WEAT effect sizes and permutation tests; 🔎 debiasing that fixes the metric but not the information; reward-model country bias |
| Sustainability | 🔎 $6ND$ → GPU-hours → energy → carbon, matching Llama 2's reported numbers |

### Exercises
1. In §2.4, add a **spurious feature** that predicts the label in-distribution but not under shift. Do models that use it fall *below* the line?
2. In §3.1, vary the SAE's dictionary size (32, 48, 64, 128) and $\lambda$. When does it start to split or merge features?
3. In §4.1, add Spanish and Hindi text to the tokenizer's training data and re-measure the cost ratios.
4. In §7.2, implement **iterative null-space projection** (Ravfogel et al. 2020: repeatedly train a linear classifier for the bias and project out its direction). How many iterations until the classifier is at chance?
5. In §8, estimate the inference footprint of serving a 70B model to a million users who each generate 1,000 tokens per day for a year.

### References
* Min et al. 2022 (*Rethinking the Role of Demonstrations*); Wei et al. 2023 (larger LMs do ICL differently); Lake & Baroni 2018 (SCAN); Kim & Linzen 2020 (COGS).
* Taori et al. 2020; Miller et al. 2021 (*Accuracy on the Line*); Gururangan et al. 2018.
* Karpathy, Johnson & Fei-Fei 2016; Hewitt, Thickstun, Manning & Liang 2023 (Backpack LMs); Elhage et al. 2022 (superposition); Bricken et al. 2023 and Cunningham et al. 2023 (sparse autoencoders).
* Adda et al. 2016; Kreutzer et al. 2022 (*Quality at a Glance*); Nekoto et al. 2020 (Masakhane); Petrov et al. 2023 (tokenizer unfairness).
* Romera-Paredes et al. 2023 (FunSearch); Taylor et al. 2022 (Galactica); Bloom 1984 (the 2-sigma problem); Alayrac et al. 2022 (Flamingo); Liu et al. 2023 (LLaVA).
* Caliskan, Bryson & Narayanan 2017 (WEAT); Bolukbasi et al. 2016 (hard debiasing); Gonen & Goldberg 2019 (*Lipstick on a Pig*); Ravfogel et al. 2020 (INLP); Ryan, Held & Yang 2024.
* Hovy & Yang 2021 (*The Importance of Modeling Social Factors of Language*); Bosselut et al. 2019 (COMET); Sap et al. 2022.
* Strubell, Ganesh & McCallum 2019 (*Energy and Policy Considerations for Deep Learning in NLP*); Touvron et al. 2023 (Llama 2, carbon reporting); Luccioni et al. 2023.